# C7 / basalis fibroblast cross-atlas marker comparison

This notebook compares the published marker sets from:

- **Fitzgerald et al. / Gargett lab** (`mmc2.xlsx`; 30 markers per C0–C15 cluster)
- **Ulrich et al. / Hammoud lab** (`pnas.2404775121.sd06.xlsx`)

The primary C7 analysis asks whether **Fitzgerald C12 (basal fibroblast)** and **Ulrich Stro7 (C7 fibroblast)** represent the same population.

It also contains an **all-by-all comparison** between every Fitzgerald cluster and every fine-grained Ulrich marker set available in Dataset S6B–S6E (endothelial, stromal, non-ciliated epithelial, immune).

### Interpretation
Overlap coefficient and Jaccard are **candidate-ranking statistics**, not automatic cell-identity calls. Shared lineage/state programs (ECM, cycling, inflammation, etc.) can create overlap without implying identical cell identity. Inspect the genes driving important matches.

### PNAS S6C cleanup
The supplied spreadsheet appears to auto-increment the `Stro3` label across consecutive rows (`3_PV STEAP4`, `3_PV STEAP5`, ...). These rows are collapsed back into a single **`3_PV STEAP4`** marker set; otherwise they become dozens of one-gene pseudo-clusters.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_input(filename):
    """Find an input file in the notebook directory or its parent."""
    candidates = [Path(filename), Path("..") / filename]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Looked in: "
        + ", ".join(str(p) for p in candidates)
    )

FITZ = find_input("mmc2.xlsx")
PNAS = find_input("pnas.2404775121.sd06.xlsx")

OUTDIR = Path("c7_marker_overlap_results")
OUTDIR.mkdir(exist_ok=True)

print("Fitzgerald markers:", FITZ)
print("Ulrich markers:", PNAS)
print("Outputs:", OUTDIR.resolve())

## 1. Load Fitzgerald marker sets

In [ ]:
# Fitzgerald supplement: one sheet per cluster, 30 published markers per sheet.

fitz_tables = {}
fitz_sets = {}

for sheet in pd.ExcelFile(FITZ).sheet_names:
    df = pd.read_excel(FITZ, sheet_name=sheet)
    df.columns = [str(c).strip() for c in df.columns]

    gene_col = next(c for c in df.columns if c.lower() == "gene")

    df = df.dropna(subset=[gene_col]).copy()
    df[gene_col] = df[gene_col].astype(str).str.strip().str.upper()

    fitz_tables[sheet] = df
    fitz_sets[sheet] = set(df[gene_col].head(30))

fitz_summary = pd.DataFrame({
    "cluster": list(fitz_sets),
    "n_markers": [len(fitz_sets[k]) for k in fitz_sets],
})

display(fitz_summary)

## 2. Load Ulrich/Hammoud fine-grained marker sets

In [ ]:
PNAS_FINE_SHEETS = [
    "6B Blood endo markers",
    "6C Stro markers",
    "6D Non-cil Epi markers",
    "6E Imm markers",
]

PNAS_PREFIX = {
    "6B Blood endo markers": "Endo",
    "6C Stro markers": "Stro",
    "6D Non-cil Epi markers": "Epi",
    "6E Imm markers": "Imm",
}

def normalize_pnas_cluster(sheet, label):
    label = str(label).strip()

    # Apparent Excel autofill artifact in S6C:
    # 3_PV STEAP4, 3_PV STEAP5, ... should be one Stro3/PV STEAP4 cluster.
    if sheet == "6C Stro markers" and re.fullmatch(r"3_PV STEAP\d+", label):
        return "3_PV STEAP4"

    return label

def pnas_cluster_number(label):
    m = re.match(r"^(\d+)", str(label))
    return int(m.group(1)) if m else 999

def pnas_sort_key(key):
    sheet, cluster = key
    return (
        PNAS_FINE_SHEETS.index(sheet),
        pnas_cluster_number(cluster),
        str(cluster),
    )

def pnas_display(key):
    sheet, cluster = key
    prefix = PNAS_PREFIX[sheet]

    parts = str(cluster).split("_", 1)
    if len(parts) == 2 and parts[0].isdigit():
        return f"{prefix}{parts[0]}: {parts[1]}"

    return f"{prefix}: {cluster}"

pnas_rows = []

for sheet in PNAS_FINE_SHEETS:
    df = pd.read_excel(PNAS, sheet_name=sheet, header=2)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(subset=["cluster", "gene"]).copy()

    df["cluster"] = df["cluster"].map(
        lambda x: normalize_pnas_cluster(sheet, x)
    )
    df["gene"] = df["gene"].astype(str).str.strip().str.upper()
    df["sheet"] = sheet

    pnas_rows.append(df)

pnas_markers = pd.concat(pnas_rows, ignore_index=True)

_grouped = (
    pnas_markers
    .groupby(["sheet", "cluster"], sort=False)["gene"]
    .apply(set)
)

pnas_keys = sorted(list(_grouped.index), key=pnas_sort_key)

pnas_fine_sets = {
    key: set(_grouped.loc[key])
    for key in pnas_keys
}

pnas_summary = pd.DataFrame([
    {
        "sheet": key[0],
        "cluster": key[1],
        "display_label": pnas_display(key),
        "n_markers": len(pnas_fine_sets[key]),
    }
    for key in pnas_keys
])

display(pnas_summary)
print(f"\nFine-grained PNAS subclusters loaded: {len(pnas_fine_sets)}")

**Why S6A is not mixed into the all-by-all heatmap:** `6A Major cell type DEGs` is a broader hierarchy than the fine subclusters in S6B–S6E. Mixing major-lineage DEG sets with subtype DEG sets makes the ranking hard to interpret. The primary comprehensive comparison below therefore uses all available **fine-grained** subclusters.

## 3. Helper functions

In [ ]:
def pairwise_marker_metrics(left_sets, right_sets):
    # Returns shared counts, overlap coefficient, Jaccard,
    # fraction of the Fitzgerald marker set recovered, and a long-form table.
    left_names = list(left_sets.keys())
    right_items = list(right_sets.items())
    right_names = [pnas_display(k) for k, _ in right_items]

    counts = pd.DataFrame(0, index=left_names, columns=right_names, dtype=int)
    overlap = pd.DataFrame(0.0, index=left_names, columns=right_names)
    jaccard = pd.DataFrame(0.0, index=left_names, columns=right_names)
    fraction_left = pd.DataFrame(0.0, index=left_names, columns=right_names)

    records = []

    for left_name, A in left_sets.items():
        A = set(A)

        for (right_key, B), right_name in zip(right_items, right_names):
            B = set(B)

            shared = A & B
            union = A | B
            n_shared = len(shared)

            oc = (
                n_shared / min(len(A), len(B))
                if min(len(A), len(B)) > 0 else np.nan
            )
            jac = (
                n_shared / len(union)
                if len(union) > 0 else np.nan
            )
            frac = (
                n_shared / len(A)
                if len(A) > 0 else np.nan
            )

            counts.loc[left_name, right_name] = n_shared
            overlap.loc[left_name, right_name] = oc
            jaccard.loc[left_name, right_name] = jac
            fraction_left.loc[left_name, right_name] = frac

            records.append({
                "Fitzgerald_cluster": left_name,
                "Ulrich_sheet": right_key[0],
                "Ulrich_raw_cluster": right_key[1],
                "Ulrich_cluster": right_name,
                "Fitzgerald_n_markers": len(A),
                "Ulrich_n_markers": len(B),
                "n_shared": n_shared,
                "overlap_coefficient": oc,
                "jaccard": jac,
                "fraction_Fitzgerald_recovered": frac,
                "shared_genes": ", ".join(sorted(shared)),
            })

    pairs = pd.DataFrame(records)

    return counts, overlap, jaccard, fraction_left, pairs


def plot_marker_heatmap(
    matrix,
    title,
    xlabel,
    ylabel,
    filename,
    colorbar_label,
    annotate_threshold=0.10,
    figsize=None,
    column_boundaries=None,
):
    # 0–1 heatmap with truly square cells; saves a 300-dpi PNG.
    n_rows, n_cols = matrix.shape

    if figsize is None:
        figsize = (
            max(12, n_cols * 0.62),
            max(8, n_rows * 0.62),
        )

    fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(
        matrix.values,
        aspect="equal",
        vmin=0,
        vmax=1,
        cmap="viridis",
    )

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(
        matrix.columns,
        rotation=90,
        fontsize=11 if n_cols <= 15 else 9,
    )

    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(
        matrix.index,
        fontsize=12,
    )

    annotation_size = 10 if n_cols <= 15 else 7

    for i in range(n_rows):
        for j in range(n_cols):
            val = matrix.iloc[i, j]
            if pd.notna(val) and val >= annotate_threshold:
                ax.text(
                    j,
                    i,
                    f"{val:.2f}",
                    ha="center",
                    va="center",
                    fontsize=annotation_size,
                    color="white" if val < 0.45 else "black",
                )

    if column_boundaries is not None:
        for x in column_boundaries:
            ax.axvline(x, linewidth=1.5, color="white")

    ax.set_title(title, fontsize=17, pad=18)
    ax.set_xlabel(xlabel, fontsize=18, labelpad=12)
    ax.set_ylabel(ylabel, fontsize=18, labelpad=12)

    cbar = fig.colorbar(
        im,
        ax=ax,
        ticks=[0, 1],
        fraction=0.035,
        pad=0.03,
    )
    cbar.set_label(colorbar_label, fontsize=17)
    cbar.ax.tick_params(labelsize=15)

    fig.tight_layout()

    output_path = OUTDIR / filename
    fig.savefig(output_path, dpi=300, bbox_inches="tight")

    print("Saved:", output_path)
    plt.show()

    return fig, ax

## 4. Fitzgerald clusters vs Ulrich stromal subclusters

In [ ]:
pnas_stromal_sets = {
    key: genes
    for key, genes in pnas_fine_sets.items()
    if key[0] == "6C Stro markers"
}

(
    stro_counts,
    stro_overlap,
    stro_jaccard,
    stro_fraction_fitz,
    stro_pairs,
) = pairwise_marker_metrics(
    fitz_sets,
    pnas_stromal_sets,
)

stro_counts.to_csv(OUTDIR / "fitzgerald_vs_ulrich_stromal_shared_counts.csv")
stro_overlap.to_csv(OUTDIR / "fitzgerald_vs_ulrich_stromal_overlap_coefficient.csv")
stro_jaccard.to_csv(OUTDIR / "fitzgerald_vs_ulrich_stromal_jaccard.csv")
stro_pairs.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_stromal_all_pairs.csv",
    index=False,
)

display(stro_overlap)

In [ ]:
plot_marker_heatmap(
    stro_overlap,
    title="Fitzgerald et al. cluster markers vs Ulrich et al. stromal markers",
    xlabel="Ulrich et al. stromal subtype (Hammoud Lab)",
    ylabel="Fitzgerald et al. cluster (Gargett Lab)",
    filename="fitzgerald_vs_ulrich_stromal_overlap_heatmap.png",
    colorbar_label="Overlap coefficient",
    annotate_threshold=0.10,
    figsize=(12, 12),
)

In [ ]:
plot_marker_heatmap(
    stro_jaccard,
    title="Fitzgerald et al. cluster markers vs Ulrich et al. stromal markers",
    xlabel="Ulrich et al. stromal subtype (Hammoud Lab)",
    ylabel="Fitzgerald et al. cluster (Gargett Lab)",
    filename="fitzgerald_vs_ulrich_stromal_jaccard_heatmap.png",
    colorbar_label="Jaccard index",
    annotate_threshold=0.10,
    figsize=(12, 12),
)

## 5. Focused C12 ↔ Stro7 analysis

In [ ]:
from matplotlib_venn import venn2

C12 = "Cluster 12"
STRO7_KEY = ("6C Stro markers", "7_C7 Fibroblast")

fitzgerald_c12 = set(fitz_sets[C12])
ulrich_stro7 = set(pnas_fine_sets[STRO7_KEY])

intersection = fitzgerald_c12 & ulrich_stro7
ulrich_only = ulrich_stro7 - fitzgerald_c12
fitzgerald_only = fitzgerald_c12 - ulrich_stro7

print(f"Ulrich et al. Stro7 markers:     {len(ulrich_stro7)}")
print(f"Fitzgerald et al. C12 markers:   {len(fitzgerald_c12)}")
print(f"Intersection:                    {len(intersection)}")
print(
    "Overlap coefficient:            "
    f"{len(intersection) / min(len(ulrich_stro7), len(fitzgerald_c12)):.3f}"
)
print(
    "Jaccard index:                  "
    f"{len(intersection) / len(ulrich_stro7 | fitzgerald_c12):.3f}"
)

all_genes = sorted(ulrich_stro7 | fitzgerald_c12)

marker_table = pd.DataFrame({"gene": all_genes})

marker_table["Ulrich et al. (Hammoud Lab) - Stro7"] = (
    marker_table["gene"].isin(ulrich_stro7)
)
marker_table["Fitzgerald et al. (Gargett Lab) - C12"] = (
    marker_table["gene"].isin(fitzgerald_c12)
)

marker_table["region"] = marker_table["gene"].map(
    lambda g:
        "intersection" if g in intersection
        else "Ulrich only" if g in ulrich_only
        else "Fitzgerald only"
)

region_order = {
    "intersection": 0,
    "Ulrich only": 1,
    "Fitzgerald only": 2,
}

marker_table["_order"] = marker_table["region"].map(region_order)
marker_table = (
    marker_table
    .sort_values(["_order", "gene"])
    .drop(columns="_order")
)

marker_csv = OUTDIR / "ulrich_stro7_vs_fitzgerald_c12_markers.csv"
marker_table.to_csv(marker_csv, index=False)

display(marker_table)
print("Saved:", marker_csv)

fig, ax = plt.subplots(figsize=(8, 8))

v = venn2(
    subsets=(ulrich_stro7, fitzgerald_c12),
    set_labels=(
        "Ulrich et al.\nStro7: C7 fibroblast\n(Hammoud lab)",
        "Fitzgerald et al.\nC12: basal fibroblast\n(Gargett lab)",
    ),
    ax=ax,
)

for region_id in ("10", "01", "11"):
    label = v.get_label_by_id(region_id)
    if label is not None:
        label.set_fontsize(20)

for label in v.set_labels:
    if label is not None:
        label.set_fontsize(15)

ax.set_title("C7+ basal fibroblast marker overlap", fontsize=20, pad=20)

fig.tight_layout()

venn_path = OUTDIR / "ulrich_stro7_vs_fitzgerald_c12_venn.png"
fig.savefig(venn_path, dpi=300, bbox_inches="tight")

print("Saved:", venn_path)
plt.show()

### C12 specificity across every fine-grained Ulrich subcluster

In [ ]:
(
    _c12_counts,
    _c12_overlap,
    _c12_jaccard,
    _c12_fraction,
    c12_all_pairs,
) = pairwise_marker_metrics(
    {C12: fitzgerald_c12},
    pnas_fine_sets,
)

c12_rank_overlap = (
    c12_all_pairs
    .sort_values(
        ["overlap_coefficient", "n_shared", "jaccard"],
        ascending=False,
    )
    .reset_index(drop=True)
)
c12_rank_overlap.index = c12_rank_overlap.index + 1
c12_rank_overlap.index.name = "rank"

c12_rank_jaccard = (
    c12_all_pairs
    .sort_values(
        ["jaccard", "n_shared", "overlap_coefficient"],
        ascending=False,
    )
    .reset_index(drop=True)
)
c12_rank_jaccard.index = c12_rank_jaccard.index + 1
c12_rank_jaccard.index.name = "rank"

c12_rank_overlap.to_csv(
    OUTDIR / "fitzgerald_C12_vs_all_ulrich_subclusters_ranked_by_overlap.csv",
    index=True,
)
c12_rank_jaccard.to_csv(
    OUTDIR / "fitzgerald_C12_vs_all_ulrich_subclusters_ranked_by_jaccard.csv",
    index=True,
)

print("=== C12 ranked by overlap coefficient ===")
display(c12_rank_overlap.head(15))

print("\n=== C12 ranked by Jaccard index ===")
display(c12_rank_jaccard.head(15))

### Inspect the Stro7 hit versus the myoepithelial runner-up

In [ ]:
MYOEPI_KEY = ("6D Non-cil Epi markers", "7_Myoepithelial")

myoepi_genes = set(pnas_fine_sets[MYOEPI_KEY])

stro7_shared = sorted(fitzgerald_c12 & ulrich_stro7)
myoepi_shared = sorted(fitzgerald_c12 & myoepi_genes)

stro7_not_myoepi = sorted(
    (fitzgerald_c12 & ulrich_stro7) - myoepi_genes
)
myoepi_not_stro7 = sorted(
    (fitzgerald_c12 & myoepi_genes) - ulrich_stro7
)
shared_by_both = sorted(
    fitzgerald_c12 & ulrich_stro7 & myoepi_genes
)

print("=== C12 ∩ Stro7 ===")
print(f"{len(stro7_shared)} genes")
print(", ".join(stro7_shared))

print("\n=== C12 ∩ Myoepithelial ===")
print(f"{len(myoepi_shared)} genes")
print(", ".join(myoepi_shared))

print("\n=== C12 genes shared with Stro7 but NOT myoepithelial ===")
print(f"{len(stro7_not_myoepi)} genes")
print(", ".join(stro7_not_myoepi))

print("\n=== C12 genes shared with myoepithelial but NOT Stro7 ===")
print(f"{len(myoepi_not_stro7)} genes")
print(", ".join(myoepi_not_stro7))

print("\n=== C12 genes shared by both ===")
print(f"{len(shared_by_both)} genes")
print(", ".join(shared_by_both))

### Check where C7-family / basalis-program genes appear in the published marker lists

In [ ]:
C7_PROGRAM = [
    "C7",
    "CXCL12",
    "CTSK",
    "CFH",
    "GPNMB",
    "SCN7A",
    "DPT",
    "ASPN",
    "SFRP1",
    "IGFBP5",
    "IGFBP6",
]

fitz_program_presence = pd.DataFrame(
    {
        cluster: {
            gene: gene in markers
            for gene in C7_PROGRAM
        }
        for cluster, markers in fitz_sets.items()
    }
).T

print("Fitzgerald marker-list presence:")
display(fitz_program_presence)

ulrich_program_presence = pd.DataFrame(
    {
        pnas_display(key): {
            gene: gene in markers
            for gene in C7_PROGRAM
        }
        for key, markers in pnas_fine_sets.items()
    }
).T

ulrich_program_presence["n_C7_program_markers"] = (
    ulrich_program_presence[C7_PROGRAM].sum(axis=1)
)

print("\nUlrich subclusters ranked by number of C7-program markers:")
display(
    ulrich_program_presence
    .sort_values("n_C7_program_markers", ascending=False)
    .head(20)
)

fitz_program_presence.to_csv(
    OUTDIR / "fitzgerald_C7_program_marker_presence.csv"
)
ulrich_program_presence.to_csv(
    OUTDIR / "ulrich_C7_program_marker_presence.csv"
)

## 6. Comprehensive all-by-all comparison

This compares **all 16 Fitzgerald clusters** against **all 37 fine-grained Ulrich/Hammoud subclusters** represented in S6B–S6E.

Both overlap coefficient and Jaccard are retained because they answer slightly different questions:

- **Overlap coefficient:** how completely the smaller marker set is contained in the larger one.
- **Jaccard:** how similar the two full marker sets are relative to their union.

For small PNAS marker sets, overlap coefficient can be high even with few genes; inspect `n_shared`, marker-set sizes, and Jaccard alongside it.

In [ ]:
(
    all_counts,
    all_overlap,
    all_jaccard,
    all_fraction_fitz,
    all_pairs,
) = pairwise_marker_metrics(
    fitz_sets,
    pnas_fine_sets,
)

all_pairs["rank_overlap"] = (
    all_pairs
    .groupby("Fitzgerald_cluster")["overlap_coefficient"]
    .rank(method="min", ascending=False)
)
all_pairs["rank_jaccard"] = (
    all_pairs
    .groupby("Fitzgerald_cluster")["jaccard"]
    .rank(method="min", ascending=False)
)

all_counts.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_all_subclusters_shared_counts.csv"
)
all_overlap.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_all_subclusters_overlap_coefficient.csv"
)
all_jaccard.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_all_subclusters_jaccard.csv"
)
all_fraction_fitz.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_all_subclusters_fraction_fitzgerald_recovered.csv"
)
all_pairs.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_all_subclusters_all_pairs.csv",
    index=False,
)

best_overlap = (
    all_pairs
    .sort_values(
        ["Fitzgerald_cluster", "overlap_coefficient", "n_shared", "jaccard"],
        ascending=[True, False, False, False],
    )
    .groupby("Fitzgerald_cluster", as_index=False)
    .head(1)
    [[
        "Fitzgerald_cluster",
        "Ulrich_cluster",
        "n_shared",
        "Ulrich_n_markers",
        "overlap_coefficient",
        "jaccard",
    ]]
    .rename(columns={
        "Ulrich_cluster": "best_by_overlap",
        "n_shared": "overlap_n_shared",
        "Ulrich_n_markers": "overlap_reference_n_markers",
        "overlap_coefficient": "best_overlap_coefficient",
        "jaccard": "jaccard_at_overlap_best",
    })
)

best_jaccard = (
    all_pairs
    .sort_values(
        ["Fitzgerald_cluster", "jaccard", "n_shared", "overlap_coefficient"],
        ascending=[True, False, False, False],
    )
    .groupby("Fitzgerald_cluster", as_index=False)
    .head(1)
    [[
        "Fitzgerald_cluster",
        "Ulrich_cluster",
        "n_shared",
        "Ulrich_n_markers",
        "jaccard",
        "overlap_coefficient",
    ]]
    .rename(columns={
        "Ulrich_cluster": "best_by_jaccard",
        "n_shared": "jaccard_n_shared",
        "Ulrich_n_markers": "jaccard_reference_n_markers",
        "jaccard": "best_jaccard",
        "overlap_coefficient": "overlap_at_jaccard_best",
    })
)

best_hits = best_overlap.merge(
    best_jaccard,
    on="Fitzgerald_cluster",
    how="outer",
)

best_hits["_cluster_num"] = (
    best_hits["Fitzgerald_cluster"]
    .str.extract(r"(\d+)")[0]
    .astype(int)
)
best_hits = (
    best_hits
    .sort_values("_cluster_num")
    .drop(columns="_cluster_num")
    .reset_index(drop=True)
)

best_hits.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_all_subclusters_best_hits.csv",
    index=False,
)

display(best_hits)

In [ ]:
group_sizes = [
    sum(key[0] == sheet for key in pnas_fine_sets)
    for sheet in PNAS_FINE_SHEETS
]
column_boundaries = [
    x - 0.5
    for x in np.cumsum(group_sizes)[:-1]
]

plot_marker_heatmap(
    all_overlap,
    title="All Fitzgerald clusters vs all Ulrich et al. fine-grained subclusters",
    xlabel="Ulrich et al. subcluster (Hammoud Lab)",
    ylabel="Fitzgerald et al. cluster (Gargett Lab)",
    filename="fitzgerald_vs_ulrich_all_subclusters_overlap_heatmap.png",
    colorbar_label="Overlap coefficient",
    annotate_threshold=0.10,
    column_boundaries=column_boundaries,
)

In [ ]:
plot_marker_heatmap(
    all_jaccard,
    title="All Fitzgerald clusters vs all Ulrich et al. fine-grained subclusters",
    xlabel="Ulrich et al. subcluster (Hammoud Lab)",
    ylabel="Fitzgerald et al. cluster (Gargett Lab)",
    filename="fitzgerald_vs_ulrich_all_subclusters_jaccard_heatmap.png",
    colorbar_label="Jaccard index",
    annotate_threshold=0.10,
    column_boundaries=column_boundaries,
)

## 7. Top-N candidate matches for every Fitzgerald cluster

These tables are useful for exploration, but should not be treated as automatic cell-type assignments. For each candidate pair, inspect:

1. lineage consistency,
2. the actual shared genes,
3. whether the shared genes are identity-informative versus generic programs,
4. contradictory markers,
5. anatomy / spatial localization where available.

In [ ]:
TOP_N = 5

top_by_overlap = (
    all_pairs
    .sort_values(
        ["Fitzgerald_cluster", "overlap_coefficient", "n_shared", "jaccard"],
        ascending=[True, False, False, False],
    )
    .groupby("Fitzgerald_cluster", as_index=False)
    .head(TOP_N)
)

top_by_jaccard = (
    all_pairs
    .sort_values(
        ["Fitzgerald_cluster", "jaccard", "n_shared", "overlap_coefficient"],
        ascending=[True, False, False, False],
    )
    .groupby("Fitzgerald_cluster", as_index=False)
    .head(TOP_N)
)


for _df in (top_by_overlap, top_by_jaccard):
    _df["_cluster_num"] = (
        _df["Fitzgerald_cluster"]
        .str.extract(r"(\d+)")[0]
        .astype(int)
    )

top_by_overlap = (
    top_by_overlap
    .sort_values(["_cluster_num", "overlap_coefficient", "n_shared"], ascending=[True, False, False])
    .drop(columns="_cluster_num")
    .reset_index(drop=True)
)

top_by_jaccard = (
    top_by_jaccard
    .sort_values(["_cluster_num", "jaccard", "n_shared"], ascending=[True, False, False])
    .drop(columns="_cluster_num")
    .reset_index(drop=True)
)

top_by_overlap.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_all_subclusters_top5_by_overlap.csv",
    index=False,
)
top_by_jaccard.to_csv(
    OUTDIR / "fitzgerald_vs_ulrich_all_subclusters_top5_by_jaccard.csv",
    index=False,
)

print("=== Top 5 by overlap coefficient ===")
display(
    top_by_overlap[
        [
            "Fitzgerald_cluster",
            "Ulrich_cluster",
            "n_shared",
            "Ulrich_n_markers",
            "overlap_coefficient",
            "jaccard",
            "shared_genes",
        ]
    ]
)

print("\n=== Top 5 by Jaccard ===")
display(
    top_by_jaccard[
        [
            "Fitzgerald_cluster",
            "Ulrich_cluster",
            "n_shared",
            "Ulrich_n_markers",
            "jaccard",
            "overlap_coefficient",
            "shared_genes",
        ]
    ]
)

## Notes

- Fitzgerald contributes 30 published markers per cluster, so the Fitzgerald side is deliberately fixed at 30.
- Ulrich/Hammoud marker-set sizes vary by subtype.
- The primary C12 ↔ Stro7 result uses the **full published Stro7 marker set** (126 genes) versus the 30 Fitzgerald C12 markers.
- The all-by-all analysis is exploratory. A missing or ambiguous match is a legitimate biological outcome; clusters are not guaranteed to have one-to-one counterparts across atlases.